<a href="https://colab.research.google.com/github/emgakii001/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emgakii001/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
!git clone https://github.com/emgakii001/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 124 (delta 38), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.83 MiB | 9.95 MiB/s, done.
Resolving deltas: 100% (38/38), done.


In [20]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


In [21]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


## 1. My lane as an ML task (type)


This lane is a supervised scoring (ranking) task. The model learns from historical page performance and assigns each page a priority score so content reviewers can focus on the highest-value opportunities first. A scoring approach is more suitable than simple classification because reviewers have limited time and need pages ranked by priority rather than a yes/no recommendation.

In [22]:
# Section 1 — confirm we have a real supervised signal to learn from
print("Task type: supervised scoring/ranking")
print("Rows (pages):", len(df))
print("Trend categories available:", df["trend_direction"].unique())

Task type: supervised scoring/ranking
Rows (pages): 30000
Trend categories available: ['down' 'stable' 'new' 'up' 'flat']


## 2. Target or proxy


I would predict a binary label, needs_review, defined as 1 if a page's trend_direction equals "down," and 0 otherwise. This label comes from an observed outcome already present in the historical data, not a rule I invented myself — it reflects how the page actually performed over the trailing 90-day window.

To predict this label, the model would be shown signals that exist independently of the label itself — for example avg_position, content_age_days, and engagement_rate. Critically, trend_direction and trend_pct are never used as features, since the label is derived directly from them; including them as inputs would let the model "see the answer" rather than learn a genuine pattern.

What I actually want to predict — "worth a refresh slot" — isn't directly measurable in this dataset. So needs_review acts as a proxy: a measurable stand-in that's related to, but not identical to, the true decision I care about.

In [23]:
# Section 2 — build the target column and separate it from candidate features
df["needs_review"] = (df["trend_direction"] == "down").astype(int)

print("Target distribution:")
print(df["needs_review"].value_counts())
print(f"\n{df['needs_review'].mean()*100:.1f}% of pages labeled as needing review")

candidate_features = ["avg_position", "content_age_days", "engagement_rate"]
print("\nCandidate features (preview):")
print(df[candidate_features].head())# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Target distribution:
needs_review
1    16262
0    13738
Name: count, dtype: int64

54.2% of pages labeled as needing review

Candidate features (preview):
   avg_position  content_age_days  engagement_rate
0          10.6               187             5.88
1          20.3               445             0.00
2          36.5               141             0.00
3           6.2               463             1.28
4          44.0               263             0.00


## 3. Success metric

I would use Precision@50 as my success metric. This fits because reviewers can only act on a limited number of pages per cycle — in this lane's context, roughly fifty refresh slots — so what matters isn't overall accuracy across all 30,000 pages, it's whether the pages at the top of the ranked list are actually the ones that need review.

Accuracy would be misleading here: since about 54% of pages are already labeled "needs review," a model that simply guessed "needs review" for every page would score ~54% accuracy without learning anything useful. Precision@50 avoids this trap because it only rewards being right about the specific pages ranked at the top — which is the exact part of the output a reviewer actually uses.

"Good" for this metric means beating a naive baseline meaningfully — the lecture's own benchmark showed a hand-written rule reaching ~24% Precision@50, while a trained model reached ~74%. A similar jump would indicate this lane is worth the model-building effort.

In [24]:
# Section 3 — demonstrate how Precision@K would be calculated
# (using a placeholder/random score for now, since no model exists yet)
import numpy as np

np.random.seed(42)
df["placeholder_score"] = np.random.rand(len(df))  # stand-in until a real model exists

def precision_at_k(dataframe, score_col, label_col, k=50):
    top_k = dataframe.sort_values(score_col, ascending=False).head(k)
    return top_k[label_col].mean()

demo_precision = precision_at_k(df, "placeholder_score", "needs_review", k=50)
print(f"Precision@50 with a RANDOM placeholder score: {demo_precision:.2f}")
print("(This is just to show the metric mechanics — a real model in later weeks should beat this.)")# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Precision@50 with a RANDOM placeholder score: 0.56
(This is just to show the metric mechanics — a real model in later weeks should beat this.)


## 4. The unit of analysis, as a real dataframe

One row = one content page. My dataframe slice shows 30,000 rows, one per page, with columns covering traffic signals (impressions_90d, clicks_90d, avg_position), engagement (engagement_rate), content metadata (content_type, content_age_days), and the target column I built in Section 2 (needs_review). Each page is a pseudonymized item belonging to one of 32 clients, identified by content_id and client_id — used only for grouping, never as predictive features.

In [25]:
# Section 4 — show the unit of analysis as a real dataframe
lane_columns = [
    "content_id", "client_id", "content_type",
    "impressions_90d", "clicks_90d", "engagement_rate",
    "avg_position", "content_age_days",
    "needs_review"
]

lane_slice = df[lane_columns]

print("Shape (rows, columns):", lane_slice.shape)
print("\nOne row = one page. Preview:")
lane_slice.head(10)# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Shape (rows, columns): (30000, 9)

One row = one page. Preview:


,content_id,client_id,content_type,impressions_90d,clicks_90d,engagement_rate,avg_position,content_age_days,needs_review
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,5.88,10.6,187,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.00,20.3,445,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.00,36.5,141,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,1.28,6.2,463,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.00,44.0,263,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,0.00,8.5,147,1
6,content_9a34b442b552,client_8722616204,keyword article,20,0,0.00,7.0,90,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,3.57,21.2,445,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,32574,29,5.88,46.0,90,1
9,content_c27558df2b0c,client_19581e27de,keyword article,1240,2,0.00,4.9,257,1


## 5. Why ML beats a fixed rule here


A fixed rule like "flag any page where avg_position is greater than 20" would catch some declining pages, but it would miss pages that still rank reasonably well while showing declining traffic, poor engagement, or aging content — the decline isn't visible in position alone. It would also misfire on the 1,205 pages where avg_position equals 0, since that value means "no position data," not a genuinely strong ranking — a single threshold has no way to tell the difference.

In this dataset, 54.2% of pages are already labeled as declining, which shows the underlying pattern is common enough across the inventory to be worth learning systematically, rather than checked with one hand-picked cutoff. The lecture's own benchmark reinforces this: a hand-written rule reached only ~24% Precision@50, while a trained model reached ~74% — a three-fold improvement that can only come from weighing several weak signals together (position, engagement, traffic trend, content age) rather than relying on any single one in isolation.

In [26]:
# Section 5 — test the single-rule claim against the data
single_rule_flag = (df["avg_position"] > 20).astype(int)

# How much overlap does this one rule have with the actual "needs_review" label?
overlap = ((single_rule_flag == 1) & (df["needs_review"] == 1)).sum()
rule_flagged_total = single_rule_flag.sum()
actual_needing_review = df["needs_review"].sum()

precision_of_rule = overlap / rule_flagged_total if rule_flagged_total > 0 else 0
recall_of_rule = overlap / actual_needing_review if actual_needing_review > 0 else 0

print(f"Pages flagged by single rule (avg_position > 20): {rule_flagged_total}")
print(f"Of those, actually needing review: {overlap}")
print(f"Precision of single rule: {precision_of_rule:.2f}")
print(f"Recall of single rule: {recall_of_rule:.2f}")
print(f"\nTotal pages actually needing review: {actual_needing_review} ({actual_needing_review/len(df)*100:.1f}%)")

Pages flagged by single rule (avg_position > 20): 8539
Of those, actually needing review: 4510
Precision of single rule: 0.53
Recall of single rule: 0.28

Total pages actually needing review: 16262 (54.2%)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.